# NYU depth-map diagnostic and optional refinement

Run in Colab with a GPU. Upload the matching Stage 1 U-Net ZIP and then the matching Stage 2 ViT ZIP. The diagnostic plots show shared reference-based depth limits and masked absolute error. Values outside the colour scale saturate, and metrics use the original predictions. White reference or error regions have no valid reference depth.

To test a model change, set RUN_REFINEMENT to True. Each model gets matched five-epoch MAE and gradient-loss runs from the same checkpoint. Compare validation depth MAE and edge-gradient MAE. KITTI gradient supervision is limited to adjacent measured pixels.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import json
import math
import re
import shutil
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
EPOCHS, PATIENCE, BATCH_SIZE = 30, 6, 4
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
import h5py

HEIGHT, WIDTH = 192, 256
MAX_DEPTH = 10.0
NAME = "nyu"
DATA = Path("/content/stage1/datasets/nyu")
DATA.mkdir(parents=True, exist_ok=True)
source = DATA / "nyu_depth_v2_labeled.mat"
if not source.is_file():
    partial = source.with_suffix(".part")
    subprocess.run(["wget", "-c", "-O", str(partial),
                    "https://horatio.cs.nyu.edu/mit/silberman/nyu_depth_v2/nyu_depth_v2_labeled.mat"],
                   check=True)
    partial.replace(source)

dataset_file = h5py.File(source, "r")
images = dataset_file["images"]
depths = dataset_file["depths"]
assert images.shape == (1449, 3, 640, 480)
assert depths.shape == (1449, 640, 480)
scenes = [
    "".join(chr(int(code)) for code in dataset_file[ref][...].ravel())
    for ref in dataset_file["scenes"][...].ravel()
]
assert len(scenes) == len(images)
records = [(f"nyu_{i:04d}", scene) for i, scene in enumerate(scenes)]

def read_pair(index):
    rgb = Image.fromarray(images[index].transpose(2, 1, 0))
    depth = Image.fromarray(depths[index].T.astype(np.float32))
    return rgb, depth

In [ ]:
from google.colab import files
from io import BytesIO
from zipfile import ZipFile

print("Upload the Stage 1 NYU Depth V2 results ZIP (20261121T211823767832Z.zip)")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Choose one Stage 1 results ZIP.")
with ZipFile(BytesIO(next(iter(uploaded.values())))) as baseline:
    manifest = json.loads(baseline.read("splits.json"))
    original_metrics = json.loads(baseline.read("metrics.json"))
    unet_weights = baseline.read("unet_best.pt")
del uploaded

expected = dict(dataset=NAME, model="U-Net", height=HEIGHT, width=WIDTH,
                max_depth_m=MAX_DEPTH, seed=SEED, epochs=EPOCHS,
                patience=PATIENCE, batch_size=BATCH_SIZE,
                training_loss="masked MAE in meters")
for key, value in expected.items():
    if original_metrics.get(key) != value:
        raise ValueError(f"Stage 1 settings mismatch for {key}. Check the uploaded ZIP.")

split_names = ("train", "validation", "test")
if set(manifest) != set(split_names):
    raise ValueError("Expected train, validation, and test splits.")
indices = {}
seen = set()
seen_groups = set()
for name in split_names:
    rows = manifest[name]
    if not rows:
        raise ValueError(f"Empty {name} split.")
    groups = set()
    indices[name] = []
    for row in rows:
        index = row["index"]
        if not 0 <= index < len(records) or records[index] != (row["sample"], row["group"]):
            raise ValueError(f"Dataset does not match the Stage 1 manifest: {row}")
        if index in seen:
            raise ValueError(f"Duplicate index in Stage 1 manifest: {index}")
        seen.add(index)
        groups.add(row["group"])
        indices[name].append(index)
    if groups & seen_groups:
        raise ValueError(f"Scene or drive overlap in {name} split.")
    seen_groups.update(groups)
if len(seen) != len(records):
    raise ValueError("The current dataset has a different number of pairs than Stage 1.")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output = Path("/content/refinement/results") / NAME / run_id
output.mkdir(parents=True)
(output / "splits.json").write_text(json.dumps(manifest, indent=2))
print({name: len(rows) for name, rows in indices.items()})

class PairedDepth(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, item):
        rgb, depth = read_pair(self.rows[item])
        rgb = rgb.resize((WIDTH, HEIGHT), Image.Resampling.BILINEAR)
        depth = depth.resize((WIDTH, HEIGHT), Image.Resampling.NEAREST)
        x = torch.from_numpy(np.array(rgb, dtype=np.float32)).permute(2, 0, 1) / 255
        y = torch.from_numpy(np.array(depth, dtype=np.float32)).unsqueeze(0)
        return x, y

datasets = {name: PairedDepth(rows) for name, rows in indices.items()}
loaders = {
    name: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=(name == "train"),
                     generator=torch.Generator().manual_seed(SEED))
    for name, ds in datasets.items()
}

In [ ]:
class UNetDepth(nn.Module):
    def __init__(self):
        super().__init__()

        def block(inputs, outputs):
            return nn.Sequential(
                nn.Conv2d(inputs, outputs, 3, padding=1), nn.ReLU(),
                nn.Conv2d(outputs, outputs, 3, padding=1), nn.ReLU(),
            )

        self.pool = nn.MaxPool2d(2)
        self.enc1 = block(3, 16)
        self.enc2 = block(16, 32)
        self.enc3 = block(32, 64)
        self.bridge = block(64, 128)
        self.dec3 = block(128 + 64, 64)
        self.dec2 = block(64 + 32, 32)
        self.dec1 = block(32 + 16, 16)
        self.head = nn.Conv2d(16, 1, 1)

    def forward(self, rgb):
        a = self.enc1(rgb)
        b = self.enc2(self.pool(a))
        c = self.enc3(self.pool(b))
        value = self.bridge(self.pool(c))
        for skip, decode in ((c, self.dec3), (b, self.dec2), (a, self.dec1)):
            value = F.interpolate(value, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            value = decode(torch.cat((value, skip), dim=1))
        return F.softplus(self.head(value))

class ViTDepth(nn.Module):
    def __init__(self):
        super().__init__()
        patch_size, dim = 16, 128
        self.grid = (HEIGHT // patch_size, WIDTH // patch_size)
        self.patch_embedding = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        self.position = nn.Parameter(torch.empty(1, math.prod(self.grid), dim))
        nn.init.trunc_normal_(self.position, std=0.02)
        block = nn.TransformerEncoderLayer(
            dim, nhead=4, dim_feedforward=4 * dim, dropout=0.1,
            activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            block, num_layers=4, norm=nn.LayerNorm(dim), enable_nested_tensor=False,
        )
        self.depth_head = nn.Sequential(
            nn.Conv2d(dim, 64, 3, padding=1),
            nn.GELU(),
            nn.Upsample(scale_factor=4, mode="bilinear", align_corners=False),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.GELU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(32, 1, 3, padding=1),
        )

    def forward(self, rgb):
        patches = self.patch_embedding(rgb)
        tokens = patches.flatten(2).transpose(1, 2)
        features = self.encoder(tokens + self.position).transpose(1, 2)
        features = features.reshape(rgb.shape[0], -1, *self.grid)
        depth = self.depth_head(features)
        return F.softplus(F.interpolate(
            depth, size=rgb.shape[-2:], mode="bilinear", align_corners=False,
        ))

unet = UNetDepth().to(device)
unet.load_state_dict(torch.load(BytesIO(unet_weights), map_location=device, weights_only=True))
unet.eval()
torch.manual_seed(SEED)

In [ ]:
from copy import deepcopy

print("Upload the Stage 2 NYU ZIP (20261121T220455708046Z.zip)")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Select one Stage 2 results ZIP.")
with ZipFile(BytesIO(next(iter(uploaded.values())))) as archive:
    settings = json.loads(archive.read("comparison.json"))
    if (settings["dataset"], settings["height"], settings["width"]) != (NAME, HEIGHT, WIDTH):
        raise ValueError("This Stage 2 archive is for a different dataset.")
    if json.loads(archive.read("splits.json")) != manifest:
        raise ValueError("Stage 1 and Stage 2 splits differ.")
    vit = ViTDepth().to(device)
    vit.load_state_dict(torch.load(BytesIO(archive.read("vit_best.pt")),
                                   map_location=device, weights_only=True))
vit.eval()
del uploaded

In [ ]:
def evaluate_epoch(network, batches, optimizer=None):
    network.train(optimizer is not None)
    sums = torch.zeros(4, dtype=torch.float64, device=device)
    count = 0
    with torch.set_grad_enabled(optimizer is not None):
        for rgb, depth in batches:
            rgb, depth = rgb.to(device), depth.to(device)
            valid = torch.isfinite(depth) & (depth > 0) & (depth <= MAX_DEPTH)
            observed = depth[valid]
            if observed.numel() == 0:
                continue
            predicted = network(rgb)[valid]
            error = (predicted - observed).abs()
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
                error.mean().backward()
                optimizer.step()
            with torch.no_grad():
                sums += torch.stack((
                    error.sum(dtype=torch.float64),
                    error.square().sum(dtype=torch.float64),
                    (error / observed).sum(dtype=torch.float64),
                    (torch.maximum(predicted / observed, observed / predicted) < 1.25).sum(),
                ))
                count += observed.numel()
    if count == 0:
        raise ValueError("No valid depth pixels.")
    mae, mse, abs_rel, delta1 = (sums / count).cpu().tolist()
    if not all(math.isfinite(value) for value in (mae, mse, abs_rel, delta1)):
        raise FloatingPointError("Non-finite evaluation metrics.")
    return dict(mae_m=mae, rmse_m=math.sqrt(mse), abs_rel=abs_rel, delta1=delta1, pixels=count)

In [ ]:
# Set to True after inspecting the original predictions to run the refinement.
RUN_REFINEMENT = False
REFINE_EPOCHS = 5
GRADIENT_WEIGHT = 0.5

def gradient_errors(prediction, target, valid):
    for axis in (-1, -2):
        if axis == -1:
            adjacent = valid[..., 1:] & valid[..., :-1]
        else:
            adjacent = valid[..., 1:, :] & valid[..., :-1, :]
        reference = torch.diff(target, dim=axis)[adjacent]
        predicted = torch.diff(prediction, dim=axis)[adjacent]
        yield (predicted - reference).abs(), reference.abs()

def fine_tune(original, gradient_weight):
    torch.manual_seed(SEED)
    network = deepcopy(original).to(device).train()
    batches = DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True,
                         generator=torch.Generator().manual_seed(SEED))
    optimizer = torch.optim.AdamW(network.parameters(), lr=5e-5, weight_decay=1e-2)
    for epoch in range(REFINE_EPOCHS):
        for rgb, target in batches:
            rgb, target = rgb.to(device), target.to(device)
            valid = torch.isfinite(target) & (target > 0) & (target <= MAX_DEPTH)
            if not valid.any():
                continue
            prediction = network(rgb)
            loss = (prediction[valid] - target[valid]).abs().mean()
            if gradient_weight:
                pairs = [error for error, _ in gradient_errors(prediction, target, valid)]
                count = sum(error.numel() for error in pairs)
                if count:
                    loss = loss + gradient_weight * sum(error.sum() for error in pairs) / count
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        print(f"Weight {gradient_weight:g} | epoch {epoch + 1}/{REFINE_EPOCHS}", flush=True)
    return network.eval()

@torch.inference_mode()
def validation_edge_mae(network):
    error_sum, count = 0.0, 0
    threshold = 0.10 if NAME == "nyu" else 0.50
    for rgb, target in loaders["validation"]:
        rgb, target = rgb.to(device), target.to(device)
        valid = torch.isfinite(target) & (target > 0) & (target <= MAX_DEPTH)
        for error, reference in gradient_errors(network(rgb), target, valid):
            selected = error[reference >= threshold]
            error_sum += selected.sum().item()
            count += selected.numel()
    return {"threshold_m": threshold, "gradient_mae_m": error_sum / count if count else None,
            "valid_edge_pairs": count}

originals = {"U-Net": unet, "ViT": vit}
variants = dict(originals)
if RUN_REFINEMENT:
    for name, network in originals.items():
        for label, weight in [("MAE control", 0.0), ("MAE + gradient", GRADIENT_WEIGHT)]:
            refined = fine_tune(network, weight)
            variants[f"{name} {label}"] = refined
            torch.save(refined.state_dict(), output / f"{name.lower()}_{label.replace(' ', '_')}.pt")

report = {}
for label, network in variants.items():
    report[label] = {
        **evaluate_epoch(network, loaders["validation"]),
        **validation_edge_mae(network),
    }
    print(label, report[label])
(output / "validation_diagnostics.json").write_text(json.dumps(report, indent=2))

In [ ]:
def compare_image(rgb, target, networks, filename):
    predictions = []
    with torch.inference_mode():
        for label, network in networks.items():
            prediction = network(rgb.unsqueeze(0).to(device))[0, 0].cpu().numpy()
            predictions.append((label, prediction))
    measured = target[0].numpy()
    valid = np.isfinite(measured) & (measured > 0) & (measured <= MAX_DEPTH)
    if not valid.any():
        raise ValueError("This frame contains no valid reference depth.")
    low, high = np.percentile(measured[valid], [2, 98])
    high = max(high, low + 0.01)
    reference = np.ma.masked_where(~valid, measured)
    errors = [np.abs(pred - measured)[valid] for _, pred in predictions]
    error_limit = max(float(np.percentile(np.concatenate(errors), 95)), 0.01)
    fig, axes = plt.subplots(2, len(predictions)+1,
                            figsize=(4*(len(predictions)+1), 6), layout="constrained")
    axes[0, 0].imshow(reference, cmap="viridis", vmin=low, vmax=high, interpolation="nearest")
    axes[0, 0].set_title("Reference depth")
    axes[1, 0].imshow(rgb.permute(1, 2, 0))
    axes[1, 0].set_title("RGB")
    for column, ((label, pred), error) in enumerate(zip(predictions, errors), start=1):
        depth_plot = axes[0, column].imshow(pred, cmap="viridis", vmin=low, vmax=high,
                                           interpolation="nearest")
        axes[0, column].set_title(label)
        error_plot = axes[1, column].imshow(np.ma.masked_where(~valid, np.abs(pred - measured)),
                                            cmap="magma", vmin=0, vmax=error_limit,
                                            interpolation="nearest")
        axes[1, column].set_title(f"MAE {error.mean():.2f} m")
    for axis in axes.flat:
        axis.axis("off")
    fig.colorbar(depth_plot, ax=axes[0].tolist(), label="Depth (m)", extend="both")
    fig.colorbar(error_plot, ax=axes[1].tolist(), label="Absolute error (m)", extend="max")
    fig.savefig(output / filename, dpi=170)
    plt.show()
    plt.close(fig)

for index in np.linspace(0, len(datasets["validation"])-1, 3, dtype=int):
    rgb, target = datasets["validation"][index]
    compare_image(rgb, target, originals, f"validation_{index:04d}_models.png")
    if RUN_REFINEMENT:
        for name in originals:
            compare_image(rgb, target,
                          {label: network for label, network in variants.items() if label.startswith(name)},
                          f"validation_{index:04d}_{name}.png")

files.download(shutil.make_archive(str(output), "zip", root_dir=output))